In [2]:
import pandas as pd
from scipy.stats import wilcoxon, mannwhitneyu, friedmanchisquare
from collections import defaultdict
import numpy as np

Statistics for comparing coil-combined to all-coil architectures. We look at three different architectures:
* Deep cascade
* E2EVarnet
* MoDL
Each of these has been implemented in a coil-combined way (as per the papers) and an all-coil way. Between all-coil and coil-combined, we effectively have paired data. We can do a Wilcoxon signed-rank test for each architecture to determine whether the impact was significant. Make it two-tailed since we don't have a super strong prior for there always being an improvement.

Set $\alpha$=0.05, and for Bonferroni correction we will do 2 (for R factor) x 3 (for 3 architectures) x 3 (for 3 metrics) - 1 (becase E2E doesn't use APD metric) = 17

In [19]:
R = 8
architecture = 'MoDL'
dataset = 'CC'
metrics = ['ssim', 'psnr', 'apd']
final_results_dict = {'architecture': [], 'style': [], 'dataset': [], 'R': [], 'ssim': [], 'ssim std': [], 'psnr': [], 'psnr std': [], 'apd': [], 'apd std': []}
final_results_path = 'results/final_results.csv'
final_results_df = pd.read_csv(final_results_path)

In [20]:
# dictionary of {architecture:{style:{R_factor}}}
if dataset == 'GBM':
    # inclusive min slice, exclusive end slice
    min_slice, max_slice = 30, 130  # nearly every volume has 160 slices ( some have slightly more or less)
    final_results = {
        'DC': {
            'all_coil': {
                4: 'results/DC_all_coils_V_v16_R=4_GBM.csv',
                8: 'results/DC_all_coils_V_v16_R=8_GBM.csv'
                },
            'coil_combine': {
                4: 'results/DC_coil_combine_V_v2_R=4_GBM.csv',
                8: 'results/DC_coil_combine_V_v2_R=8_GBM.csv'
                }         
                            },
        'E2Evarnet': {
            'all_coil': {
                4: 'results/E2Evarnet_all_coils_V_v15_R=4_GBM.csv',
                8: 'results/E2Evarnet_all_coils_V_v15_R=8_GBM.csv'
            },
            'coil_combine': {
                4: 'results/E2Evarnet_coil_combine_V_v5_resumed_R=4_GBM.csv',
                8: 'results/E2Evarnet_coil_combine_V_v5_resumed_R=8_GBM.csv'
            }
        },
        'MoDL':{
            'all_coil': {
                4: 'results/MoDL_all_coils_V_v0_resumed_R=4_GBM.csv',
                8: 'results/MoDL_all_coils_V_v0_resumed_R=8_GBM.csv',
            },
            'coil_combine': {
                4: 'results/MoDL_coil_combine_V_v0_R=4_GBM.csv',
                8: 'results/MoDL_coil_combine_V_v0_R=8_GBM.csv',
            }
        }
    }
else:
    min_slice, max_slice = 78, 178
    final_results = {
        'DC': {
            'all_coil': {
                4: 'results/DC_all_coils_V_v16_R=4_CC.csv',
                8: 'results/DC_all_coils_V_v16_R=8_CC.csv'
                },
            'coil_combine': {
                4: 'results/DC_coil_combine_V_v2_R=4_CC.csv',
                8: 'results/DC_coil_combine_V_v2_R=8_CC.csv'
                }         
                            },
        'E2Evarnet': {
            'all_coil': {
                4: 'results/E2Evarnet_all_coils_V_v15_R=4_CC.csv',
                8: 'results/E2Evarnet_all_coils_V_v15_R=8_CC.csv'
            },
            'coil_combine': {
                4: 'results/E2Evarnet_coil_combine_V_v5_resumed_R=4_CC.csv',
                8: 'results/E2Evarnet_coil_combine_V_v5_resumed_R=8_CC.csv'
            }
        },
        'MoDL':{
            'all_coil': {
                4: 'results/MoDL_all_coils_V_v0_resumed_R=4_CC.csv',
                8: 'results/MoDL_all_coils_V_v0_resumed_R=8_CC.csv',
            },
            'coil_combine': {
                4: 'results/MoDL_coil_combine_V_v0_R=4_CC.csv',
                8: 'results/MoDL_coil_combine_V_v0_R=8_CC.csv',
            }
        }
    }

all_coil_results = pd.read_csv(final_results[architecture]['all_coil'][R])
coil_combine_results = pd.read_csv(final_results[architecture]['coil_combine'][R])

In [21]:
final_results_dict['architecture'].extend([architecture, architecture])
final_results_dict['style'].extend(['all_coil', 'coil_combine'])
final_results_dict['dataset'].extend([dataset, dataset])
final_results_dict['R'].extend([R, R])

alpha = 0.05 / 12

for metric in metrics:
    # select relevant slices
    all_coils_results = all_coil_results.loc[
        (all_coil_results['slice_num'] >= min_slice) & 
        (all_coil_results['slice_num'] < max_slice)
        , :]

    # pull out patient id (per volume)
    all_coil_results['p_id'] = all_coil_results['img_id'].str.split('_s').str[0]
    coil_combine_results['p_id'] = coil_combine_results['img_id'].str.split('_s').str[0]

    # group by volume
    all_coil_volume_grouped = all_coil_results.groupby(by='p_id')
    coil_combine_volume_grouped = coil_combine_results.groupby(by='p_id')

    # get mean per volume
    all_coil_metric = all_coil_volume_grouped[metric].mean().to_numpy()
    coil_combine_metric = coil_combine_volume_grouped[metric].mean().to_numpy()
    
    # get mean and standard deviation of metric per volume
    all_coil_mean, all_coil_std = np.mean(all_coil_metric), np.std(all_coil_metric)
    coil_combine_mean, coil_combine_std = np.mean(coil_combine_metric), np.std(coil_combine_metric)

    print(f'All coil {metric}: {all_coil_mean:.2f} +- {all_coil_std:.3f}')
    print(f'Coil combine {metric}: {coil_combine_mean:.2f} +- {coil_combine_std:.3f}')

    try:
        res = wilcoxon(all_coil_metric, coil_combine_metric, alternative='two-sided') 
        w_pval = res.pvalue
        if w_pval < alpha:
            print(f'They ARE significantly different, p={w_pval:.4f}\n')
        else:
            print(f'They ARE NOT significantly different, p={w_pval:.4f}\n')
    except ValueError:
        pass

    # update final results
    final_results_dict[metric].append(all_coil_mean)
    final_results_dict[f'{metric} std'].append(all_coil_std)
    final_results_dict[metric].append(coil_combine_mean)
    final_results_dict[f'{metric} std'].append(coil_combine_std)

final_results_update = pd.DataFrame(final_results_dict)
final_results_update = pd.concat([final_results_df, final_results_update], ignore_index=True)
final_results_update.drop_duplicates(inplace=True, ignore_index=True, keep='last', subset=['architecture', 'style', 'dataset', 'R'])  # keep updated values (updates were concatenated to end of final results)
final_results_update.to_csv(final_results_path, index=False)

All coil ssim: 0.90 +- 0.009
Coil combine ssim: 0.91 +- 0.008
They ARE significantly different, p=0.0000

All coil psnr: 32.56 +- 1.027
Coil combine psnr: 32.56 +- 0.902
They ARE NOT significantly different, p=0.2223

All coil apd: 0.09 +- 0.008
Coil combine apd: 0.09 +- 0.007
They ARE significantly different, p=0.0000

